# Week model v1 — freeze-time lags

The live week pack is dominated by `demand_lag_48` (yesterday). On a Monday freeze, yesterday has not happened yet for Thursday–Sunday, so that slot is filled with **monthly climatology** (~29 GW for an October Friday lunch). This week’s lunches were ~21 GW. The tree followed the fake lag.

This notebook retrains the **same** week architecture (LightGBM mean, then residual P10/P50/P90, same `LGB_WEEK` settings) with lags built **as at freeze time**:

- lead &lt; 24h: keep real yesterday (`lag_48`)
- lead ≥ 24h: copy last week (`lag_336`) into `lag_48`

Day and next-30 packs are unchanged. Artefacts are written to `notebooks/week/v1/models`, **not** `data/models`. Production keeps the old week pack until you copy this folder over and the metadata flag `freeze_time_lags` is present (live `run_pack` then applies the same mask).

Run from this folder or the repo root. `prepare_history()` uses the Previous Runs cache; the fit is the slow cell.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "energy_forecast").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
HERE = ROOT / "notebooks" / "week" / "v1"
OUT_DIR = HERE / "models"
sys.path.insert(0, str(ROOT / "src"))

from energy_forecast.evaluate import as_board_scored, board_naive_scale, overall_metrics, score, time_series_kfold
from energy_forecast.features import apply_freeze_time_lags
from energy_forecast.forecast import prepare_history
from energy_forecast.live import WEEK, frozen_dir, load_board, load_context_actuals
from energy_forecast.model import (
    LGB_WEEK,
    WEEK_N_ESTIMATORS,
    load_models,
    predict_quantiles,
    split_training_frame,
    train_models,
)
from energy_forecast.modes import MEDIUM, resample_to_mode
from energy_forecast.review import _attach_frozen_and_weather, _load_archive_weather, _pack_rows
from energy_forecast.tune import naive_scale_for_product

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
print("repo", ROOT)
print("write models to", OUT_DIR)
print("week trees", WEEK_N_ESTIMATORS, LGB_WEEK)
prod = load_models()

repo D:\energy-forecast
write models to D:\energy-forecast\notebooks/week/v1\models
week trees 450 {'learning_rate': 0.022965408930055223, 'num_leaves': 19, 'min_child_samples': 61, 'subsample': 0.781673112232677, 'colsample_bytree': 0.6385450911486689}


## 1. What the last Monday freeze actually fed the tree

No training yet. Patch `lag_48` on the stored week freeze and re-score the **current** production trees. That is the serve-only fill (last week instead of climatology).

In [2]:
paths = sorted(frozen_dir().glob("week_*.parquet"))
print("week freezes", [p.name for p in paths])
prod = load_models()
board = load_board()
ctx = load_context_actuals()
joined = _attach_frozen_and_weather(
    WEEK,
    _pack_rows(WEEK, board, ctx).dropna(subset=["actual_mw", "p50"]),
    _load_archive_weather(),
)
joined = joined.copy()
joined["timestamp"] = pd.to_datetime(joined["timestamp"], utc=True)
if "region_id" not in joined.columns:
    joined["region_id"] = "gb"
for column in prod.features:
    if column not in joined.columns:
        joined[column] = 0.0

patched = apply_freeze_time_lags(joined)
issued = predict_quantiles(joined, prod)
as_freeze = predict_quantiles(patched, prod)
compare = pd.DataFrame(
    {
        "timestamp": joined["timestamp"],
        "lead_hours": pd.to_numeric(joined["lead_hours"], errors="coerce"),
        "actual_mw": pd.to_numeric(joined["actual_mw"], errors="coerce"),
        "issued_p50": issued["p50"],
        "freeze_lag_p50": as_freeze["p50"],
        "lag_48_issued": pd.to_numeric(joined["demand_lag_48"], errors="coerce"),
        "lag_48_freeze": pd.to_numeric(patched["demand_lag_48"], errors="coerce"),
        "lag_336": pd.to_numeric(joined.get("demand_lag_336"), errors="coerce"),
    }
)
long = compare["lead_hours"] >= 24
print(
    "this freeze n", len(compare),
    "issued MAE", (compare["actual_mw"] - compare["issued_p50"]).abs().mean(),
    "if lag_48 were last week MAE", (compare["actual_mw"] - compare["freeze_lag_p50"]).abs().mean(),
)
print("leads ≥ 24h n", int(long.sum()),
      "issued MAE", (compare.loc[long, "actual_mw"] - compare.loc[long, "issued_p50"]).abs().mean(),
      "freeze-lag MAE", (compare.loc[long, "actual_mw"] - compare.loc[long, "freeze_lag_p50"]).abs().mean())
noon = compare.loc[(pd.to_datetime(compare["timestamp"], utc=True).dt.tz_convert("Europe/London").dt.hour == 12)
                   & (pd.to_datetime(compare["timestamp"], utc=True).dt.tz_convert("Europe/London").dt.minute == 0)]
noon.assign(day=pd.to_datetime(noon["timestamp"], utc=True).dt.tz_convert("Europe/London").dt.strftime("%a %d %b"))

week freezes ['week_20260920T2300.parquet', 'week_20260927T2300.parquet', 'week_20261004T2300.parquet']
this freeze n 336 issued MAE 1690.424986571405 if lag_48 were last week MAE 1365.095401589962
leads ≥ 24h n 288 issued MAE 1795.8644189139914 freeze-lag MAE 1416.3132364356413


,timestamp,lead_hours,actual_mw,issued_p50,freeze_lag_p50,lag_48_issued,lag_48_freeze,lag_336,day
24,2026-09-28 11:00:00+00:00,12.0,"23,633.0","25,017.4","25,017.4","22,923.0","22,923.0","22,295.0",Mon 28 Sep
72,2026-09-29 11:00:00+00:00,36.0,"24,830.0","25,944.5","25,757.1","24,528.3","23,132.0","23,132.0",Tue 29 Sep
120,2026-09-30 11:00:00+00:00,60.0,"22,541.0","25,602.8","24,790.1","26,016.0","23,572.0","23,572.0",Wed 30 Sep
168,2026-10-01 11:00:00+00:00,84.0,"20,996.0","26,208.9","23,311.2","28,879.5","21,447.0","21,447.0",Thu 01 Oct
216,2026-10-02 11:00:00+00:00,108.0,"20,762.0","25,519.1","23,750.8","28,691.3","24,045.0","24,045.0",Fri 02 Oct
264,2026-10-03 11:00:00+00:00,132.0,"20,908.0","22,473.5","19,542.1","24,456.0","16,845.0","16,845.0",Sat 03 Oct
312,2026-10-04 11:00:00+00:00,156.0,"19,277.0","23,011.3","22,016.2","25,346.6","22,923.0","22,923.0",Sun 04 Oct


## 2. History with freeze-time lags

Same Previous Runs frame as notebook 03, then the lag mask. Holdout is the last eight weeks, 6-hour display grid, as for the medium pack.

In [3]:
history = prepare_history()
freeze_history = apply_freeze_time_lags(history)
train, holdout = split_training_frame(freeze_history)
train_true, holdout_true = split_training_frame(history)
print("history", len(history), "freeze-time", len(freeze_history))
print("train", len(train), train["timestamp"].min(), "->", train["timestamp"].max())
print("holdout", len(holdout), holdout["timestamp"].min(), "->", holdout["timestamp"].max())
long = pd.to_numeric(history["lead_hours"], errors="coerce") >= 24
sample = history.loc[long, ["timestamp", "lead_hours", "demand_lag_48", "demand_lag_336"]].head(3)
sample["lag_48_after"] = freeze_history.loc[sample.index, "demand_lag_48"]
sample

history 384368 freeze-time 384368
train 363624 2024-01-01 00:00:00+00:00 -> 2026-08-04 23:00:00+01:00
holdout 20744 2026-08-04 23:30:00+01:00 -> 2026-09-29 23:30:00+01:00


,timestamp,lead_hours,demand_lag_48,demand_lag_336,lag_48_after
1,2024-01-01 00:00:00+00:00,24.0,"22,232.0","18,694.0","18,694.0"
2,2024-01-01 00:00:00+00:00,48.0,"22,232.0","18,694.0","18,694.0"
3,2024-01-01 00:00:00+00:00,72.0,"22,232.0","18,694.0","18,694.0"


## 3. Current production on this holdout

True yesterday (what notebook 03 reported) vs freeze-time lags (what a Monday freeze can actually know). Production trees, no refit.

In [4]:
scale = naive_scale_for_product(train_true, "medium")

def pack_metrics(frame, models, label):
    scored = resample_to_mode(score(frame, models), MEDIUM)
    out = overall_metrics(scored, scale_mse=scale)
    out.insert(0, "model", label)
    return out

baseline = pd.concat(
    [
        pack_metrics(holdout_true, prod, "production / true lag_48"),
        pack_metrics(holdout, prod, "production / freeze-time lag_48"),
    ],
    ignore_index=True,
)
baseline

,model,split,n,mae,rmse,mape,bias,pinball_10,pinball_50,pinball_90,coverage_80,interval_width,skill_vs_climatology,rmsse,wrmsse
0,production / true lag_48,all,1728,903.6,"1,211.7",0.0,-136.2,223.1,451.8,211.6,0.9,"3,869.6",0.4,0.5,0.4
1,production / freeze-time lag_48,all,1728,"1,156.3","1,553.1",0.1,-50.3,271.8,578.1,268.7,0.8,"3,800.0",0.2,0.6,0.6


## 4. Train week v1

Writes `notebooks/week/v1/models`. Does not overwrite `data/models`.

In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
v1, metrics = train_models(freeze_history, model_dir=OUT_DIR, role="week")
meta_path = OUT_DIR / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))
meta["freeze_time_lags"] = True
meta["lag_48_policy"] = (
    "If lead_hours >= 24, demand_lag_48 is demand_lag_336 (known at freeze). "
    "Not true yesterday and not monthly climatology."
)
meta["parent_role"] = "week"
meta["update"] = "v1"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("saved", OUT_DIR)
print("holdout rows from train_models")
metrics

saved D:\energy-forecast\notebooks/week/v1\models
holdout rows from train_models


,metric,quantile,value
0,pinball,0.1,258.9
1,pinball,0.5,574.9
2,mae,0.5,"1,149.9"
3,pinball,0.9,251.0
4,coverage_80,0.8,0.8
5,interval_scale,0.8,1.1


## 5. Compare on freeze-time holdout

v1 should be judged on freeze-time lags. Production on true lags is the old (optimistic) number.

In [6]:
from energy_forecast.model import load_models as load_from

v1 = load_from(OUT_DIR)
compared = pd.concat(
    [
        pack_metrics(holdout_true, prod, "production / true lag_48"),
        pack_metrics(holdout, prod, "production / freeze-time lag_48"),
        pack_metrics(holdout, v1, "v1 / freeze-time lag_48"),
    ],
    ignore_index=True,
)
compared

,model,split,n,mae,rmse,mape,bias,pinball_10,pinball_50,pinball_90,coverage_80,interval_width,skill_vs_climatology,rmsse,wrmsse
0,production / true lag_48,all,1728,903.6,"1,211.7",0.0,-136.2,223.1,451.8,211.6,0.9,"3,869.6",0.4,0.5,0.4
1,production / freeze-time lag_48,all,1728,"1,156.3","1,553.1",0.1,-50.3,271.8,578.1,268.7,0.8,"3,800.0",0.2,0.6,0.6
2,v1 / freeze-time lag_48,all,1728,"1,043.7","1,377.7",0.0,-188.1,231.6,521.8,233.5,0.8,"3,926.8",0.3,0.5,0.5


In [7]:
imp = pd.Series(
    v1.mean_boosters["gb"].feature_importance(importance_type="gain"),
    index=list(v1.features),
).sort_values(ascending=False)
print("v1 mean-tree gain")
imp.head(12).to_frame("gain")

v1 mean-tree gain


,gain
demand_lag_336,"105,101,373,191,968.0"
demand_lag_48,"94,241,470,877,024.0"
settlement_period,"17,485,424,577,632.0"
month_cos,"13,569,324,630,176.0"
shortwave_radiation,"5,647,419,733,504.0"
period_sin,"4,474,957,586,816.0"
cdd_sum_72h,"4,130,146,501,760.0"
day_of_week,"4,071,363,712,416.0"
temp_mean_fwd_12h,"3,010,491,910,048.0"
hour,"2,772,306,721,920.0"


## 5b. Time-series k-fold

Shuffled k-fold would leak future demand into training. This is **expanding-window** validation on unique timestamps (same idea as sklearn `TimeSeriesSplit`): fold 1 trains on the earliest data and tests the next block; fold 2 trains on everything up to that block, and so on. Previous Runs leads 0–7 for the same half-hour stay in the same fold.

Run it on the **train** split only so the last eight weeks stay a true holdout. Each fold fits the week trees once (`refit_full=False`). Five folds is several times the cost of section 4. Set `RUN_KFOLD = False` to skip.

`embargo_days=7` is optional if you want `lag_336` labels out of the training window; `0` matches live (last week is known at freeze).

In [8]:
RUN_KFOLD = True
N_SPLITS = 5
EMBARGO_DAYS = 0  # 7 if you want a week gap before each test block

if RUN_KFOLD:
    kfold = time_series_kfold(
        train,
        n_splits=N_SPLITS,
        n_estimators=WEEK_N_ESTIMATORS,
        mean_params=LGB_WEEK,
        residual_params=LGB_WEEK,
        eval_mode=MEDIUM,
        embargo_days=EMBARGO_DAYS,
        refit_full=False,
    )
else:
    print("skipped time-series k-fold")
    kfold = None
kfold

D:\energy-forecast\src\energy_forecast\evaluate.py:366: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([table, pd.DataFrame([mean_row])], ignore_index=True)


,split,fold,train_end,test_start,test_end,n_train,n,mae,rmse,mape,bias,pinball_10,pinball_50,pinball_90,coverage_80,interval_width,skill_vs_climatology,rmsse,wrmsse
0,fold 1/5,0,2024-06-06 20:30:00+00:00,2024-06-06 21:00:00+00:00,2024-11-11 16:00:00+00:00,60624,5056,"1,227.7","1,636.5",0.1,361.5,345.9,613.8,324.3,0.9,"5,928.9",NaN,0.6,0.6
1,fold 2/5,1,2024-11-11 16:00:00+00:00,2024-11-11 16:30:00+00:00,2025-04-18 11:30:00+00:00,121224,5048,"1,340.5","1,810.0",0.0,16.6,405.8,670.3,332.1,0.9,"6,172.6",0.4,0.8,0.8
2,fold 3/5,2,2025-04-18 11:30:00+00:00,2025-04-18 12:00:00+00:00,2025-09-23 07:00:00+00:00,181824,5056,"1,037.3","1,393.8",0.0,-319.1,268.7,518.6,295.1,0.9,"5,132.5",0.3,0.5,0.5
3,fold 4/5,3,2025-09-23 07:00:00+00:00,2025-09-23 07:30:00+00:00,2026-02-28 02:30:00+00:00,242424,5056,"1,222.6","1,634.3",0.0,21.9,316.4,611.3,286.1,0.9,"4,876.3",0.3,0.6,0.6
4,fold 5/5,4,2026-02-28 02:30:00+00:00,2026-02-28 03:00:00+00:00,2026-08-04 22:00:00+00:00,303024,5056,"1,208.4","1,631.4",0.1,-287.1,270.5,604.2,288.1,0.8,"4,224.8",0.3,0.6,0.6
5,mean,None,None,None,None,181824,25272,"1,207.3","1,621.2",NaN,-41.2,NaN,NaN,NaN,0.9,"5,267.0",NaN,NaN,0.6


## 5c. Final check: last completed week (live WRMSSE)

The 0.96 WRMSSE on the dashboard is **this Monday–Monday freeze**, scored on half-hours against a 56-day weekly naive — not the 6-hour holdout in section 5. This cell uses the same window (`week_review_span`) and the same 30-minute WRMSSE scale so the numbers sit next to that 0.96.

- **Issued** is what the board already published (climatology `lag_48`).
- **Production + freeze-time lags** is the same trees with last-week `lag_48`.
- **v1 + freeze-time lags** is the retrained pack on those same frozen weather/lead rows.

The freeze parquet does **not** store `hour` / `month` / period sins / 72h trajectory. Re-scoring must rebuild those (`_ensure_calendar` + weather path). Filling missing features with `0` is not a valid v1 score.

If v1 models are not on disk yet, run section 4 first.

In [9]:
from energy_forecast.forecast import predict_quantiles_recursive
from energy_forecast.model import load_models as load_from
from energy_forecast.regions import SERIES_ID
from energy_forecast.review import (
    _ensure_calendar,
    _ready_predict_frame,
    _recompute_weather_features,
    week_review_span,
)
from energy_forecast.settlement import LONDON_TZ

prod = load_models()
start, end = week_review_span(load_board(), load_context_actuals())
print("completed week", pd.Timestamp(start).tz_convert(LONDON_TZ), "->", pd.Timestamp(end).tz_convert(LONDON_TZ))

board = load_board()
ctx = load_context_actuals()
joined = _attach_frozen_and_weather(
    WEEK,
    _pack_rows(WEEK, board, ctx).dropna(subset=["actual_mw", "p50"]),
    _load_archive_weather(),
)
joined = joined.copy()
joined["timestamp"] = pd.to_datetime(joined["timestamp"], utc=True)
start_u, end_u = pd.to_datetime(start, utc=True), pd.to_datetime(end, utc=True)
week = joined.loc[(joined["timestamp"] >= start_u) & (joined["timestamp"] < end_u)].copy()
week = week.reset_index(drop=True)
week["region_id"] = SERIES_ID
print("scored half-hours", len(week.dropna(subset=["actual_mw", "p50"])))
print(
    "frozen missing (were filled with 0 before)",
    [c for c in prod.features if c not in week.columns],
)

scale_live = board_naive_scale()
print("WRMSSE scale is 56-day 30-min naive, same as the live board")


def with_features(frame, models):
    # Frozen week files omit hour/month/sins. Filling those with 0 made v1 look
    # like a 26 GW midnight-in-January forecast. Rebuild calendar; fill only
    # missing T-path columns so Monday freeze weather stays as issued.
    out = _ensure_calendar(frame.copy().reset_index(drop=True))
    out["region_id"] = SERIES_ID
    work = out.drop(columns=["window_start"], errors="ignore")
    rebuilt = _recompute_weather_features(work)
    keys = ["timestamp"]
    if "lead_hours" in out.columns:
        keys = ["timestamp", "lead_hours"]
    take = [c for c in models.features if c in rebuilt.columns]
    right = rebuilt.loc[:, list(dict.fromkeys([*keys, *take]))].drop_duplicates(keys, keep="last")
    merged = out.merge(right, on=keys, how="left", suffixes=("", "_rebuilt"))
    for column in models.features:
        rebuilt_col = f"{column}_rebuilt"
        if column not in merged.columns and rebuilt_col in merged.columns:
            merged[column] = merged[rebuilt_col]
        elif rebuilt_col in merged.columns:
            merged[column] = pd.to_numeric(merged[column], errors="coerce").combine_first(
                pd.to_numeric(merged[rebuilt_col], errors="coerce")
            )
    merged["region_id"] = SERIES_ID
    return _ready_predict_frame(merged, models)


def metrics_row(frame, label, climatology=None):
    scored = as_board_scored(frame, climatology=climatology)
    out = overall_metrics(scored, scale_mse=scale_live)
    out.insert(0, "model", label)
    return out, scored


issued_tbl, issued_scored = metrics_row(week, "issued (board)", prod.climatology)
rows = [issued_tbl]

patched = apply_freeze_time_lags(with_features(week, prod))
prod_hat = predict_quantiles(patched, prod)
prod_week = week.copy()
prod_week["p50"] = prod_hat["p50"].to_numpy()
prod_week["p10"] = prod_hat["p10"].to_numpy()
prod_week["p90"] = prod_hat["p90"].to_numpy()
prod_tbl, _ = metrics_row(prod_week, "production + freeze-time lags", prod.climatology)
rows.append(prod_tbl)

rec_in = with_features(week, prod).copy()
if "demand_mw" in rec_in.columns:
    rec_in["demand_mw"] = float("nan")
rec_hat = predict_quantiles_recursive(rec_in, prod).sort_values("timestamp")
rec_week = week.sort_values("timestamp").copy()
rec_week["p50"] = rec_hat["p50"].to_numpy()
rec_week["p10"] = rec_hat["p10"].to_numpy()
rec_week["p90"] = rec_hat["p90"].to_numpy()
rec_tbl, _ = metrics_row(rec_week, "production + recursive P50 lags", prod.climatology)
rows.append(rec_tbl)
print("Tue lunch lag_48 issued vs recursive", float(rec_in.loc[rec_in.index[48 + 24], "demand_lag_48"]) if len(rec_in) > 72 else None, float(rec_hat.iloc[48 + 24]["demand_lag_48"]))

v1_scored = None
if (OUT_DIR / "metadata.json").exists():
    v1_live = load_from(OUT_DIR)
    v1_in = apply_freeze_time_lags(with_features(week, v1_live))
    v1_hat = predict_quantiles(v1_in, v1_live)
    v1_week = week.copy()
    v1_week["p50"] = v1_hat["p50"].to_numpy()
    v1_week["p10"] = v1_hat["p10"].to_numpy()
    v1_week["p90"] = v1_hat["p90"].to_numpy()
    v1_tbl, v1_scored = metrics_row(v1_week, "v1 + freeze-time lags", v1_live.climatology)
    rows.append(v1_tbl)
else:
    print("no v1 artefacts yet — run section 4")

latest_week = pd.concat(rows, ignore_index=True)
print("board WRMSSE this week was ~0.96; issued row should sit next to that")
latest_week

completed week 2026-09-28 00:00:00+01:00 -> 2026-10-05 00:00:00+01:00
scored half-hours 336
WRMSSE scale is 56-day 30-min naive, same as the live board
board WRMSSE this week was ~0.96; issued row should sit next to that


,model,split,n,mae,rmse,mape,bias,pinball_10,pinball_50,pinball_90,coverage_80,interval_width,skill_vs_climatology,rmsse,wrmsse
0,issued (board),all,336,"1,518.4","2,114.8",0.1,-971.5,305.2,759.2,298.8,0.7,"3,943.9",0.0,0.9,0.8
1,production + freeze-time lags,all,336,"1,365.1","1,631.4",0.1,-853.9,196.3,682.5,323.1,0.9,"4,947.5",0.0,0.7,0.7
2,v1 + freeze-time lags,all,336,"2,632.0","3,004.3",0.1,"-2,460.1",649.4,"1,316.0",515.5,0.4,"5,264.5",0.0,1.3,1.2


In [10]:
local = pd.to_datetime(week["timestamp"], utc=True).dt.tz_convert(LONDON_TZ)
daily = pd.DataFrame(
    {
        "day": local.dt.strftime("%a %d %b"),
        "actual": pd.to_numeric(week["actual_mw"], errors="coerce"),
        "issued": pd.to_numeric(week["p50"], errors="coerce"),
        "prod_freeze": pd.to_numeric(prod_week["p50"], errors="coerce"),
        "recursive": pd.to_numeric(rec_week["p50"], errors="coerce"),
    }
)
if v1_scored is not None:
    daily["v1"] = pd.to_numeric(v1_week["p50"], errors="coerce").to_numpy()
rows = []
for day, g in daily.groupby("day", sort=False):
    row = {
        "day": day,
        "n": int(len(g)),
        "actual": float(g["actual"].mean()),
        "issued": float(g["issued"].mean()),
        "issued MAE": float((g["actual"] - g["issued"]).abs().mean()),
        "prod freeze": float(g["prod_freeze"].mean()),
        "prod freeze MAE": float((g["actual"] - g["prod_freeze"]).abs().mean()),
        "recursive": float(g["recursive"].mean()),
        "recursive MAE": float((g["actual"] - g["recursive"]).abs().mean()),
    }
    if "v1" in g.columns:
        row["v1"] = float(g["v1"].mean())
        row["v1 MAE"] = float((g["actual"] - g["v1"]).abs().mean())
    rows.append(row)
pd.DataFrame(rows)

,day,n,actual,issued,issued MAE,v1,v1 MAE
0,Mon 28 Sep,48,"25,068.0","24,930.2",732.5,"26,635.8","1,906.3"
1,Tue 29 Sep,48,"25,370.5","25,136.5",586.0,"27,417.4","2,132.8"
2,Wed 30 Sep,48,"24,487.1","24,803.3","1,300.4","26,891.7","2,624.2"
3,Thu 01 Oct,48,"24,114.3","26,064.9","2,014.3","26,978.2","2,955.9"
4,Fri 02 Oct,48,"22,979.6","25,584.7","2,610.2","26,712.3","3,732.7"
5,Sat 03 Oct,48,"22,621.7","22,829.7","1,001.1","24,248.9","1,867.8"
6,Sun 04 Oct,48,"21,441.6","23,533.7","2,384.4","24,419.1","3,204.3"


## 6. Promote (optional)

Only after you are happy with the holdout table. This copies v1 over the live week pack. Day and hour folders are not touched. Then restart the local server and re-issue the week freeze.

Leave this cell un-run if you only want the side-by-side artefacts.

In [11]:
PROMOTE = False  # set True to replace data/models week artefacts
if PROMOTE:
    import shutil
    from energy_forecast.paths import DATA_MODELS

    for name in ("metadata.json", "climatology.parquet", "metrics.csv"):
        src = OUT_DIR / name
        if src.exists():
            shutil.copy2(src, DATA_MODELS / name)
    for src in OUT_DIR.glob("*.txt"):
        shutil.copy2(src, DATA_MODELS / src.name)
    print("copied week v1 into", DATA_MODELS)
    print("restart uvicorn and re-issue the week pack")
else:
    print("skipped promote; artefacts remain in", OUT_DIR)

skipped promote; artefacts remain in D:\energy-forecast\notebooks/week/v1\models
